# Диагностика допуска Scanner, 22 октября 2026

Источник: Dzammbo/Scaner. Срез последнего обновления: 22 октября, 16:29 МСК. Производственные правила не изменены.

Единица анализа: один исходный вход на матч; ставка 1; исходный тотал и коэффициент; расчёт азиатских четвертных линий. Незакрытые и изначально неоценённые входы исключены из ROI. Отобранные подгруппы не пополняются шестым матчем или более поздним входом.

Текущая основная серия: 167 рассчитанных входов. Исторический контроль: 324 первых зафиксированных входа той же версии v8 до запуска новой серии 21 октября, 10:21 МСК. Более ранние версии правил не смешиваются с ними. Исторический порядок сбора и выбора отличается от нынешнего общего Top5: это диагностическое сравнение, не точный повтор текущей стратегии и не нетронутый тест.

Код и входные пути закреплены рядом. Для воспроизведения в репозитории нужна исследовательская ветка `research/selection-lab-20261018`, Python, NumPy и SciPy.

In [1]:
from pathlib import Path
import importlib.util, json
here = Path.cwd()
script = here / 'selection-diagnosis.py'
spec = importlib.util.spec_from_file_location('selection_diagnosis', script)
analysis = importlib.util.module_from_spec(spec)
spec.loader.exec_module(analysis)
roots = [here, *here.parents]
repo = next((p for p in roots if (p / analysis.REPORT_PATH).exists()), None)
report_path = repo / analysis.REPORT_PATH if repo else here / 'diagnostic-current.json'
points_path = repo / analysis.POINTS_PATH if repo else here / 'diagnostic-points.json.gz'
result = analysis.analyze(report_path, points_path)
saved = json.loads((here / 'selection-diagnosis.json').read_text())
assert result['sources'] == saved['sources']
assert result['filters'] == saved['filters']
assert result['overlap'] == saved['overlap']
print('Проверено: уникальность входов, соединение с исходными котировками, независимый Decimal-расчёт, сумма подгрупп, совпадение пересчёта с сохранённым результатом.')
print('Текущая серия:', result['filters']['current_primary']['baseline']['n'], 'входов')
print('Исторический контроль:', result['filters']['earlier_recorded']['baseline']['n'], 'входа')


Проверено: уникальность входов, соединение с исходными котировками, независимый Decimal-расчёт, сумма подгрупп, совпадение пересчёта с сохранённым результатом.
Текущая серия: 167 входов
Исторический контроль: 324 входа


## Проверенные ограничения

| Условие | Исторический контроль: N / ROI / без топ-3 | Текущая серия: N / ROI / без топ-3 |
|---|---|---|
| Исходный допуск | 324 / -4.69% / -5.72% (N=321) | 167 / +0.23% / -1.64% (N=164) |
| Без 00:00-08:00 МСК | 248 / -1.08% / -2.38% (N=245) | 143 / +6.42% / +4.36% (N=140) |
| Голевой профиль <1,10 | 260 / +0.88% / -0.35% (N=257) | 111 / +5.74% / +3.08% (N=108) |
| Оба ограничения | 203 / +3.83% / +2.33% (N=200) | 94 / +9.22% / +6.17% (N=91) |
| Не менее 2 ударов за 5 минут | 176 / -9.89% / -11.87% (N=173) | 89 / +16.56% / +13.59% (N=86) |
| Только места 1-3 | 243 / -5.82% / -7.19% (N=240) | 156 / +3.25% / +1.31% (N=153) |
| Нужно ещё 2 гола | 62 / +9.78% / +4.89% (N=59) | 18 / +1.32% / - |

Голевой профиль - среднее число голов в матчах двух команд в своей лиге, делённое на средний тотал лиги. 1,10 означает результативность на 10% выше этой базы. Это историческая характеристика, не вероятность выигрыша текущего тотала.

Час относится ко времени входа, не к началу матча. Ограничения выявлены после просмотра результатов: все они исследовательские. Дробные пороги 1,05 / 1,10 / 1,15 проверены как чувствительность и сохранены полностью; лучший соседний порог не выбран.

In [2]:
for name in ('earlier_recorded', 'current_primary'):
    print(name)
    for key in ('night_00_08', 'profile_at_least_1_1', 'one_shot5'):
        v = result['loss_groups'][name][key]
        print(key, 'N=', v['n'], 'net=', round(v['net_units'], 4), 'ROI=', round(v['roi_pct'], 2))
    print('Пересечение признаков:')
    for key, v in result['overlap'][name].items():
        print(key, 'N=', v['n'], 'net=', round(v['net_units'], 4), 'ROI=', round(v['roi_pct'], 2))


earlier_recorded
night_00_08 N= 76 net= -12.5125 ROI= -16.46
profile_at_least_1_1 N= 64 net= -17.4875 ROI= -27.32
one_shot5 N= 148 net= 2.2125 ROI= 1.49
Пересечение признаков:
night=False,high_profile=False N= 203 net= 7.775 ROI= 3.83
night=False,high_profile=True N= 45 net= -10.4625 ROI= -23.25
night=True,high_profile=False N= 57 net= -5.4875 ROI= -9.63
night=True,high_profile=True N= 19 net= -7.025 ROI= -36.97
current_primary
night_00_08 N= 24 net= -8.7875 ROI= -36.61
profile_at_least_1_1 N= 56 net= -5.9875 ROI= -10.69
one_shot5 N= 78 net= -14.35 ROI= -18.4
Пересечение признаков:
night=False,high_profile=False N= 94 net= 8.6625 ROI= 9.22
night=False,high_profile=True N= 49 net= 0.5125 ROI= 1.05
night=True,high_profile=False N= 17 net= -2.2875 ROI= -13.46
night=True,high_profile=True N= 7 net= -6.5 ROI= -92.86


## Что обнаружено в механике

Все 186 матчей с сохранённой точной котировкой, сильным допуском v8 и минутой 8-60 после начала серии в итоге есть в основном журнале. Это совпадение по матчам за период, не утверждение о выборе всех сильных в каждом отдельном цикле. Из 109 циклов с новыми выборами только 5 набрали полную пятёрку. За пределами сохранённого обогащённого массива полнота неизвестна.

Главный предмет проверки - допуск. В `live_ranking.py` метка «сильный» означает достаточность данных и качественное наличие свежих ударов/продолжения давления. Оценки вероятности прохода тотала в ней нет. Все допущенные имеют класс 0 и пятиминутное окно; цена и величина тотала не входят в спортивный порядок. Исторический голевой профиль служит лишь поздним критерием при равенстве более важных признаков; нельзя приписать ему все проигрыши его подгруппы.

Пример из сохранённого среза: Harju - Flora, 14-я минута, 2:0. За последние пять минут один удар мимо, ноль в створ; в предыдущем доступном двухминутном отрезке ещё один удар той же команды. Вход получил «сильный», ТБ 5,0 за 1,925; итог 3:1 и проигрыш. Пример иллюстрирует смысл допуска, не доказывает прибыльность запрета таких входов.

Действующий отдельный фильтр модельного EV ≥0,05 также не улучшил текущую серию: N=90, ROI −0,875%, без трёх лучших −4,4109%. Механически включать его как обязательный фильтр оснований нет.

In [3]:
capacity = result['ranking_capacity']
assert capacity['observed_exact_priced_strong_8_60_unique'] == 186
assert capacity['strong_observed_never_selected_ids'] == []
print(json.dumps(capacity, ensure_ascii=False, indent=2))
for name in ('earlier_recorded', 'current_primary'):
    v = result['filters'][name]['no_night_and_profile_below_1_1']
    print(name, 'По дням:', json.dumps(v['days_msk'], ensure_ascii=False))
    print('Без лучшего дня:', v['without_best_day'])
    print('Описательный 95% bootstrap:', v['descriptive_iid_match_bootstrap_95_pct'])


{
  "selected_unique_including_open_and_unpriced": 186,
  "observed_exact_priced_strong_8_60_unique": 186,
  "strong_observed_never_selected_ids": [],
  "selected_without_any_observed_priced_strong_ids": [],
  "cycles_by_number_selected": {
    "1": 66,
    "3": 15,
    "5": 5,
    "4": 2,
    "2": 21
  },
  "settled_entries_with_an_earlier_saved_strong_quote": 5,
  "earlier_quote_delay_rows": [
    {
      "id": "12049970",
      "original_profit": -1.0,
      "earlier_profit": -1.0,
      "delay_minutes": 10.06599231561025
    },
    {
      "id": "12049974",
      "original_profit": 0.48750000000000004,
      "earlier_profit": 0.0,
      "delay_minutes": 10.06599231561025
    },
    {
      "id": "12051801",
      "original_profit": -1.0,
      "earlier_profit": -1.0,
      "delay_minutes": 9.729553099473318
    },
    {
      "id": "13326887",
      "original_profit": -0.5,
      "earlier_profit": -1.0,
      "delay_minutes": 8.25389883518219
    },
    {
      "id": "13217790",
  

## Практический кандидат и границы вывода

Для параллельной проверки на новых матчах: прежний общий Top5, затем входы 08:00-24:00 МСК и исторический профиль <1,10; без пополнения исключённых матчей и без переноса точки входа. В текущей серии остаются 94 из 167 входов, то есть 56,3% объёма. ROI +9,22%, без трёх лучших +6,17%. В историческом контроле N=203, ROI +3,83%, без трёх лучших +2,33%.

Это улучшение на просмотренной истории, его устойчивость не доказана. Исторический вариант без своего лучшего дня даёт −3,74% (N=100); текущий без 21 октября даёт −8,50% (N=10). Описательный 95%-интервал текущего ROI примерно −8,2...+26,7%. Он дополнительно не учитывает подбор фильтров и зависимость внутри лиг/дней. Сопоставление профилей при одинаковом счёте, линии и близких минуте/цене оставляет мало пар; его нельзя считать установлением причины.

Ночь в основном представлена американскими турнирами. Час может быть признаком состава лиг, а не самостоятельной причиной убытка. Ночные и высокопрофильные входы пересекаются, их убытки нельзя складывать без разбиения.

Фильтр двух ударов сейчас выглядит особенно прибыльным, но ухудшает более ранний контроль. Его не следует превращать в обязательное правило по результатам одного удачного дня. Наличие голов/ударов и выгодность предложенной линии - разные оценки; улучшение спортивной активности само по себе не подтверждает положительного ожидания.

Основные правила не изменены. В `selection-diagnosis.json` сохранены все проверенные варианты, дневные результаты, пересечения, чувствительность порога, подбор сопоставимых пар и 167 исходных строк текущей серии с признаком прохождения кандидата.